# 🧤 BridgeSpeak — BiLSTM NLP Pipeline
### Generative AI & NLP Project | B.Tech CSE 2025-26

This notebook builds the ML pipeline for BridgeSpeak:
1. **NLP Preprocessing** — Tokenization, Stopword Removal, Lemmatization (NLTK)
2. **Word Embeddings** — Word2Vec
3. **BiLSTM Classifier** — Urgency + Intent Detection
4. **Export** — Save model for TensorFlow.js (browser integration)

**Concepts covered:** Ch2 (NLP Pipeline), Ch3 (Word2Vec), Ch8 (BiLSTM), Ch12 (GenAI integration)

## 📦 Step 1 — Install & Import Libraries

In [ ]:
# Install required libraries
!pip install tensorflowjs gensim nltk -q

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

import numpy as np
import pandas as pd
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from gensim.models import Word2Vec
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

print('✅ All libraries imported successfully!')
print(f'TensorFlow version: {tf.__version__}')

## 📝 Step 2 — Dataset

We create a labeled dataset of sign language gloss inputs.

**Labels:**
- `urgency`: urgent / not_urgent
- `intent`: medical / request / social / emergency

> In a real project, this would be collected from actual users. For demo purposes, we create a representative dataset.

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
# Format: (text, urgency_label, intent_label)
# urgency: 'urgent' | 'not_urgent'
# intent:  'medical' | 'request' | 'social' | 'emergency'

raw_data = [
    # ── MEDICAL / NOT URGENT ────────────────────────────────────────────────
    ("head pain mild", "not_urgent", "medical"),
    ("doctor appointment need", "not_urgent", "medical"),
    ("medicine take time", "not_urgent", "medical"),
    ("fever low feel", "not_urgent", "medical"),
    ("hospital visit tomorrow", "not_urgent", "medical"),
    ("check blood pressure", "not_urgent", "medical"),
    ("prescription refill need", "not_urgent", "medical"),
    ("cold cough slight", "not_urgent", "medical"),
    ("eye pain small", "not_urgent", "medical"),
    ("stomach ache little", "not_urgent", "medical"),
    ("sleep problem night", "not_urgent", "medical"),
    ("allergy medicine ask", "not_urgent", "medical"),
    ("back pain sit long", "not_urgent", "medical"),
    ("throat sore drink warm", "not_urgent", "medical"),
    ("dentist appointment schedule", "not_urgent", "medical"),

    # ── MEDICAL / URGENT ────────────────────────────────────────────────────
    ("chest pain bad now", "urgent", "medical"),
    ("breathing problem serious", "urgent", "medical"),
    ("head pain very bad", "urgent", "medical"),
    ("heart beat fast wrong", "urgent", "medical"),
    ("blood much bleeding", "urgent", "medical"),
    ("cannot breathe help", "urgent", "medical"),
    ("fever very high shaking", "urgent", "medical"),
    ("pain chest arm", "urgent", "medical"),
    ("dizzy falling cannot stand", "urgent", "medical"),
    ("swallow cannot throat close", "urgent", "medical"),
    ("unconscious almost feel", "urgent", "medical"),
    ("medicine overdose took", "urgent", "medical"),
    ("wound deep bleeding stop not", "urgent", "medical"),
    ("seizure happening now", "urgent", "medical"),
    ("allergic reaction swelling", "urgent", "medical"),

    # ── REQUEST / NOT URGENT ────────────────────────────────────────────────
    ("water want please", "not_urgent", "request"),
    ("food hungry want", "not_urgent", "request"),
    ("bathroom need go", "not_urgent", "request"),
    ("sit rest want", "not_urgent", "request"),
    ("phone call make", "not_urgent", "request"),
    ("window open please", "not_urgent", "request"),
    ("light off please", "not_urgent", "request"),
    ("blanket cold need", "not_urgent", "request"),
    ("charger phone need", "not_urgent", "request"),
    ("directions lost need", "not_urgent", "request"),
    ("paper pen write", "not_urgent", "request"),
    ("chair sit need", "not_urgent", "request"),
    ("juice drink want", "not_urgent", "request"),
    ("quiet please loud", "not_urgent", "request"),
    ("repeat please understand not", "not_urgent", "request"),
    ("slow speak please", "not_urgent", "request"),
    ("write please cannot hear", "not_urgent", "request"),
    ("help carry bag", "not_urgent", "request"),
    ("door open cannot", "not_urgent", "request"),
    ("time tell please", "not_urgent", "request"),

    # ── SOCIAL / NOT URGENT ─────────────────────────────────────────────────
    ("hello good morning", "not_urgent", "social"),
    ("thank you very much", "not_urgent", "social"),
    ("nice meet you", "not_urgent", "social"),
    ("bye goodbye see later", "not_urgent", "social"),
    ("sorry excuse me", "not_urgent", "social"),
    ("how are you", "not_urgent", "social"),
    ("fine good thank", "not_urgent", "social"),
    ("happy today feel", "not_urgent", "social"),
    ("sad feel today", "not_urgent", "social"),
    ("name what you", "not_urgent", "social"),
    ("welcome please come", "not_urgent", "social"),
    ("congratulations well done", "not_urgent", "social"),
    ("wait please moment", "not_urgent", "social"),
    ("understand yes no", "not_urgent", "social"),
    ("agree disagree think", "not_urgent", "social"),

    # ── EMERGENCY ───────────────────────────────────────────────────────────
    ("help emergency now", "urgent", "emergency"),
    ("fire danger run", "urgent", "emergency"),
    ("accident happened call ambulance", "urgent", "emergency"),
    ("someone fell unconscious", "urgent", "emergency"),
    ("danger threat unsafe feel", "urgent", "emergency"),
    ("call police now", "urgent", "emergency"),
    ("lost child cannot find", "urgent", "emergency"),
    ("earthquake shaking building", "urgent", "emergency"),
    ("flood water rising fast", "urgent", "emergency"),
    ("hit attack someone hurt", "urgent", "emergency"),
    ("gas smell leak danger", "urgent", "emergency"),
    ("child drowning water", "urgent", "emergency"),
    ("robbery theft happening now", "urgent", "emergency"),
    ("electric shock happened", "urgent", "emergency"),
    ("ambulance call immediately", "urgent", "emergency"),
]

df = pd.DataFrame(raw_data, columns=['text', 'urgency', 'intent'])
print(f'✅ Dataset created: {len(df)} samples')
print(f'\nUrgency distribution:\n{df["urgency"].value_counts()}')
print(f'\nIntent distribution:\n{df["intent"].value_counts()}')
df.head(10)

## 🔧 Step 3 — NLP Pipeline (Ch 2 + Ch 1)

Applying the full 6-step NLP pipeline:
1. Tokenization
2. Lowercasing
3. Stopword Removal
4. Lemmatization
5. Vectorization (next step — Word2Vec)
6. Model Training

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Keep these — they carry meaning in sign language gloss input
KEEP_WORDS = {'not', 'no', 'cannot', 'help', 'need', 'want', 'bad', 'good'}
stop_words -= KEEP_WORDS

def preprocess(text):
    """
    Full NLP Pipeline:
    1. Lowercase
    2. Tokenize
    3. Remove stopwords (keep important ones)
    4. Lemmatize
    """
    # Step 1: Lowercase
    text = text.lower()

    # Step 2: Tokenize
    tokens = word_tokenize(text)

    # Step 3: Remove stopwords + punctuation
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]

    # Step 4: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return tokens

# Apply pipeline to full dataset
df['tokens'] = df['text'].apply(preprocess)

# Show before/after
print('── NLP Pipeline Demo ─────────────────────────────────────')
for _, row in df.head(5).iterrows():
    print(f'  Raw   : "{row["text"]}')
    print(f'  Tokens: {row["tokens"]}')
    print()

## 📐 Step 4 — Word2Vec Embeddings (Ch 3)

Training Word2Vec on our corpus so similar words get similar vectors.
e.g. 'pain' and 'hurt' should be close in vector space.

In [ ]:
EMBEDDING_DIM = 64   # vector size per word
MAX_SEQ_LEN  = 10   # max tokens per input

# Train Word2Vec on our token corpus
w2v_model = Word2Vec(
    sentences=df['tokens'].tolist(),
    vector_size=EMBEDDING_DIM,
    window=3,
    min_count=1,
    workers=4,
    epochs=100,
    sg=1  # Skip-gram
)

# Build vocabulary index
vocab = list(w2v_model.wv.key_to_index.keys())
word2idx = {w: i+2 for i, w in enumerate(vocab)}  # 0=pad, 1=unknown
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1
VOCAB_SIZE = len(word2idx)

# Build embedding matrix for Keras
embedding_matrix = np.zeros((VOCAB_SIZE, EMBEDDING_DIM))
for word, idx in word2idx.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f'✅ Word2Vec trained on {len(vocab)} unique words')
print(f'   Vocabulary size : {VOCAB_SIZE}')
print(f'   Embedding dim   : {EMBEDDING_DIM}')

# Demo — show similar words
print('\n── Semantic Similarity Demo (Word2Vec) ───────────────────')
for word in ['pain', 'help', 'water']:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=3)
        print(f'  Words similar to "{word}": {[(w,round(s,2)) for w,s in similar]}')

## 🔢 Step 5 — Encode & Pad Sequences

In [ ]:
def encode(tokens):
    """Convert token list to integer indices."""
    return [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>

# Encode all sequences
X = pad_sequences(
    [encode(t) for t in df['tokens']],
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)

# Encode labels
urgency_encoder = LabelEncoder()
intent_encoder  = LabelEncoder()

y_urgency = urgency_encoder.fit_transform(df['urgency'])  # 0=not_urgent, 1=urgent
y_intent  = intent_encoder.fit_transform(df['intent'])    # 0=emergency,1=medical,2=request,3=social

NUM_INTENT_CLASSES = len(intent_encoder.classes_)

print(f'✅ Sequences encoded and padded')
print(f'   X shape          : {X.shape}')
print(f'   Urgency classes  : {list(urgency_encoder.classes_)}')
print(f'   Intent classes   : {list(intent_encoder.classes_)}')
print(f'   Sample encoded   : {X[0]} → urgency={df["urgency"][0]}, intent={df["intent"][0]}')

# Train/test split (80/20)
X_train, X_test, yu_train, yu_test, yi_train, yi_test = train_test_split(
    X, y_urgency, y_intent, test_size=0.2, random_state=42, stratify=y_urgency
)
print(f'\n   Train samples    : {len(X_train)}')
print(f'   Test samples     : {len(X_test)}')

## 🧠 Step 6 — BiLSTM Model (Ch 8)

**Architecture:**
- Embedding layer (pre-trained Word2Vec weights)
- Bidirectional LSTM — reads input left→right AND right→left simultaneously
- Two output heads — one for urgency (binary), one for intent (4-class)

**Why BiLSTM?**
"chest pain bad" — the word "bad" at the end changes the urgency of "chest pain" before it.
A forward-only LSTM would process "bad" too late. BiLSTM reads both directions so every word gets full context.

In [ ]:
def build_bilstm_model(vocab_size, embedding_dim, embedding_matrix,
                        max_seq_len, num_intent_classes):
    """
    Multi-output BiLSTM:
      Input → Embedding → BiLSTM → GlobalMaxPool
                                      ↓           ↓
                               urgency_out   intent_out
    """
    inp = Input(shape=(max_seq_len,), name='input')

    # Pre-trained Word2Vec embeddings (trainable=True allows fine-tuning)
    emb = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True,
        name='word2vec_embedding'
    )(inp)

    # Bidirectional LSTM — core of the model
    # merge_mode='concat' → combines forward + backward outputs
    bilstm = Bidirectional(
        LSTM(64, return_sequences=True, dropout=0.3),
        merge_mode='concat',
        name='bilstm'
    )(emb)

    # Global max pooling — takes the most important feature across time steps
    pooled = GlobalMaxPooling1D(name='global_max_pool')(bilstm)
    dropped = Dropout(0.4)(pooled)

    # Output head 1 — Urgency (binary classification)
    urgency_out = Dense(1, activation='sigmoid', name='urgency')(dropped)

    # Output head 2 — Intent (multi-class classification)
    intent_out  = Dense(num_intent_classes, activation='softmax', name='intent')(dropped)

    model = Model(inputs=inp, outputs=[urgency_out, intent_out])

    model.compile(
        optimizer='adam',
        loss={
            'urgency': 'binary_crossentropy',
            'intent' : 'sparse_categorical_crossentropy'
        },
        metrics={
            'urgency': 'accuracy',
            'intent' : 'accuracy'
        }
    )
    return model

model = build_bilstm_model(
    VOCAB_SIZE, EMBEDDING_DIM, embedding_matrix,
    MAX_SEQ_LEN, NUM_INTENT_CLASSES
)

model.summary()

## 🏋️ Step 7 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    X_train,
    {'urgency': yu_train, 'intent': yi_train},
    validation_data=(X_test, {'urgency': yu_test, 'intent': yi_test}),
    epochs=100,
    batch_size=8,
    callbacks=[early_stop],
    verbose=1
)

print('\n✅ Training complete!')

## 📊 Step 8 — Evaluation & Visualisation

In [ ]:
# ── Training Curves ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Urgency accuracy
axes[0].plot(history.history['urgency_accuracy'], label='Train', color='#38bdf8')
axes[0].plot(history.history['val_urgency_accuracy'], label='Validation', color='#f59e0b')
axes[0].set_title('Urgency Classification Accuracy', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Intent accuracy
axes[1].plot(history.history['intent_accuracy'], label='Train', color='#38bdf8')
axes[1].plot(history.history['val_intent_accuracy'], label='Validation', color='#f59e0b')
axes[1].set_title('Intent Classification Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved')

In [ ]:
# ── Confusion Matrices ───────────────────────────────────────────────────────
urgency_preds, intent_preds = model.predict(X_test, verbose=0)

urgency_pred_labels = (urgency_preds > 0.5).astype(int).flatten()
intent_pred_labels  = np.argmax(intent_preds, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Urgency confusion matrix
cm_urgency = confusion_matrix(yu_test, urgency_pred_labels)
sns.heatmap(cm_urgency, annot=True, fmt='d', ax=axes[0],
            xticklabels=urgency_encoder.classes_,
            yticklabels=urgency_encoder.classes_,
            cmap='Blues')
axes[0].set_title('Urgency — Confusion Matrix', fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Intent confusion matrix
cm_intent = confusion_matrix(yi_test, intent_pred_labels)
sns.heatmap(cm_intent, annot=True, fmt='d', ax=axes[1],
            xticklabels=intent_encoder.classes_,
            yticklabels=intent_encoder.classes_,
            cmap='Oranges')
axes[1].set_title('Intent — Confusion Matrix', fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Classification reports
print('── Urgency Classification Report ──────────────────────────')
print(classification_report(yu_test, urgency_pred_labels,
                             target_names=urgency_encoder.classes_))

print('── Intent Classification Report ───────────────────────────')
print(classification_report(yi_test, intent_pred_labels,
                             target_names=intent_encoder.classes_))

## 🧪 Step 9 — Live Inference Test

In [ ]:
def predict(text):
    """
    Full pipeline inference:
    raw text → preprocess → encode → pad → BiLSTM → labels
    """
    tokens  = preprocess(text)
    encoded = encode(tokens)
    padded  = pad_sequences([encoded], maxlen=MAX_SEQ_LEN, padding='post')

    urgency_prob, intent_probs = model.predict(padded, verbose=0)

    urgency_label = urgency_encoder.classes_[int(urgency_prob[0] > 0.5)]
    urgency_conf  = float(urgency_prob[0]) if urgency_label == 'urgent' else 1 - float(urgency_prob[0])

    intent_idx    = np.argmax(intent_probs[0])
    intent_label  = intent_encoder.classes_[intent_idx]
    intent_conf   = float(intent_probs[0][intent_idx])

    return {
        'input'        : text,
        'tokens'       : tokens,
        'urgency'      : urgency_label,
        'urgency_conf' : round(urgency_conf * 100, 1),
        'intent'       : intent_label,
        'intent_conf'  : round(intent_conf * 100, 1)
    }

# Test samples
test_inputs = [
    "water thirst drink",
    "chest pain bad cannot breathe",
    "thank you very much",
    "help emergency ambulance call",
    "doctor appointment tomorrow need",
    "dizzy falling feel",
    "quiet please loud outside",
]

print('── Live Inference Results ──────────────────────────────────')
for text in test_inputs:
    r = predict(text)
    urgent_flag = '🚨' if r['urgency'] == 'urgent' else '✅'
    print(f"{urgent_flag} Input   : \"{r['input']}\"")
    print(f"   Tokens  : {r['tokens']}")
    print(f"   Urgency : {r['urgency']} ({r['urgency_conf']}%)")
    print(f"   Intent  : {r['intent']} ({r['intent_conf']}%)")
    print()

## 💾 Step 10 — Export for TensorFlow.js (Browser Integration)

In [ ]:
import tensorflowjs as tfjs

# Save Keras model
model.save('bridgespeak_bilstm.keras')

# Convert to TensorFlow.js format
tfjs.converters.save_keras_model(model, 'bridgespeak_tfjs')
print('✅ TF.js model saved to bridgespeak_tfjs/')

# Save all metadata the HTML app needs
metadata = {
    'word2idx'         : word2idx,
    'urgency_classes'  : list(urgency_encoder.classes_),
    'intent_classes'   : list(intent_encoder.classes_),
    'max_seq_len'      : MAX_SEQ_LEN,
    'stop_words'       : list(stop_words),
    'keep_words'       : list(KEEP_WORDS),
    'version'          : '1.0'
}

with open('bridgespeak_metadata.json', 'w') as f:
    json.dump(metadata, f)

print('✅ Metadata saved to bridgespeak_metadata.json')
print('\n── Files to download ────────────────────────────────────')
print('  📁 bridgespeak_tfjs/     → folder (model.json + weights)')
print('  📄 bridgespeak_metadata.json → vocabulary & config')
print('\nDownload both and place alongside bridgespeak.html')

In [ ]:
# Zip everything for easy download
!zip -r bridgespeak_model.zip bridgespeak_tfjs/ bridgespeak_metadata.json

# Download
from google.colab import files
files.download('bridgespeak_model.zip')
print('✅ Download started!')

## 📋 Summary

| Component | What We Built | Syllabus Chapter |
|---|---|---|
| NLP Pipeline | Tokenization → Stopword removal → Lemmatization | Ch 2 |
| Word Embeddings | Word2Vec (skip-gram, 64-dim) | Ch 3 |
| BiLSTM Classifier | Urgency detection + Intent classification | Ch 8 |
| Multi-output Model | Two heads — binary + 4-class | Ch 4 (ANN) |
| Export | TensorFlow.js for browser integration | — |

**Next step:** Load `bridgespeak_tfjs/` and `bridgespeak_metadata.json` into the BridgeSpeak HTML app.

The HTML app will:
1. Load the BiLSTM model in the browser via TensorFlow.js
2. Run preprocessing + classification locally (no server)
3. Inject urgency + intent labels into the Gemini prompt
4. Generate a natural, context-aware sentence